# Real-world optimization studio

This notebook solves the same practical mixed, multi-objective, black-box, and TeX/SCIP workflows exposed by `qqa example` and the Universal dashboard. Large tensor operations automatically use a Colab GPU when available.

In [ ]:
%pip install -q "qqa[scip]"

In [ ]:
import json

import qqa

device = qqa.resolve_device("auto")
qqa.fix_seed(7)
print("device:", device)
print("applications:", qqa.APPLICATIONS)

## 1. Mixed microgrid dispatch

Binary commitments, continuous generator outputs, integer storage units, and continuous demand response are coupled by demand, reserve, minimum-output, maximum-output, and storage constraints.

In [ ]:
dispatch = qqa.build_microgrid_dispatch()
dispatch_result = dispatch.solve(sol_size=512, num_epochs=1200, device=device, verbose=False)
dispatch_result.score

## 2. Cost × emissions × resilience

A single parallel run assigns a low-discrepancy reference direction to every replica. The adaptive augmented Lagrangian targets feasibility, while the live archive keeps nondominated plans.

In [ ]:
planning = qqa.build_microgrid_pareto()
front = planning.solve_pareto(sol_size=1024, num_epochs=1200, device=device, seed=7, verbose=False)
knee = front.select()
print("Pareto plans:", len(front.solutions))
print(
    "recommended knee:",
    dict(zip(front.objective_names, front.objectives[knee].tolist(), strict=True)),
)
qqa.plot_pareto(front, show=True)
qqa.plot_pareto_diagnostics(front, show=True)

In [ ]:
# Objectives and named decisions are aligned and export-ready.
front.to_frame(planning).head()

## 3. Risk × return × turnover portfolio

This is a mixed cardinality-constrained allocation: binary selection is coupled to continuous weights, full investment, minimum holdings, and maximum concentration. The three-objective surface includes non-convex selection changes that a single weighted-sum solve can miss.

In [ ]:
portfolio = qqa.build_portfolio_pareto()
allocations = portfolio.solve_pareto(
    sol_size=1024,
    num_epochs=1200,
    device=device,
    seed=11,
    verbose=False,
)
recommended = allocations.select(weights=[0.45, 0.35, 0.20])
print(dict(zip(allocations.objective_names, allocations.objectives[recommended].tolist(), strict=True)))
print(portfolio.score_summary(allocations.solutions[recommended]))
qqa.plot_pareto(allocations, title="Portfolio risk × return × turnover", show=True)

In [ ]:
# Audit feasibility, penalty adaptation, archive growth, and restart events.
qqa.plot_pareto_diagnostics(allocations, show=True)
allocations.to_frame(portfolio).head()

## 4. Constrained process black-box optimization

The simulator uses scalar Python logic and exposes no gradients. Expected improvement, probability of feasibility, a trust region, and parallel batches use a strict experiment budget.

In [ ]:
process = qqa.build_process_blackbox()
campaign = process.solve(budget=96, batch_size=8, workers=8, device=device, seed=7)
print(campaign.best_value, campaign.feasible, campaign.best_point)
qqa.plot_blackbox(campaign, show=True)

In [ ]:
# Resume later without evaluating any previous point again.
extended = process.solve(
    budget=128,
    batch_size=8,
    workers=8,
    device=device,
    seed=7,
    resume_from=campaign,
)
print(extended.evaluations, extended.best_value)

## 5. Audited production model → QQA → SCIP

In production, save the model returned by `qqa tex --file production.tex --output-model audited.json`, review it, and solve the same JSON offline. No API credential is present in the model.

In [ ]:
spec = qqa.ModelSpec.from_dict(
    {
        "name": "regional-production-plan",
        "variables": [
            {"name": "open_a", "kind": "binary", "lower": 0, "upper": 1, "size": 1},
            {"name": "open_b", "kind": "binary", "lower": 0, "upper": 1, "size": 1},
            {"name": "lots_a", "kind": "integer", "lower": 0, "upper": 12, "size": 1},
            {"name": "lots_b", "kind": "integer", "lower": 0, "upper": 10, "size": 1},
            {"name": "overtime", "kind": "real", "lower": 0, "upper": 16, "size": 1},
        ],
        "objectives": [
            {
                "name": "weekly_cost",
                "direction": "min",
                "expression": "1400*open_a+1100*open_b+460*lots_a+510*lots_b+38*square(overtime)",
                "unit": "USD",
            }
        ],
        "constraints": [
            {
                "name": "demand",
                "expression": "8*lots_a+7*lots_b+overtime",
                "sense": ">=",
                "rhs": 105,
                "weight": 1000,
                "scale": 105,
                "tolerance": 0.05,
            },
            {
                "name": "link_a",
                "expression": "lots_a-12*open_a",
                "sense": "<=",
                "rhs": 0,
                "weight": 500,
                "scale": 12,
                "tolerance": 0.01,
            },
            {
                "name": "link_b",
                "expression": "lots_b-10*open_b",
                "sense": "<=",
                "rhs": 0,
                "weight": 500,
                "scale": 10,
                "tolerance": 0.01,
            },
        ],
        "notes": "Two plants, integer lots, continuous overtime, activation links.",
    }
)
exact = qqa.solve_spec_scip(
    spec,
    qqa_kwargs={"sol_size": 256, "num_epochs": 1000, "device": device, "verbose": False},
    time_limit=60,
)
print(exact.objective_value, exact.scip_status, exact.gap)
exact.score

### Live TeX translation from the CLI

Keep the key in the environment or enter it with `getpass`; never put it in a notebook cell or Git:

```bash
export QQA_LLM_API_KEY='…'
qqa tex --file production.tex --solver auto --device auto \
  --show-model --output-model audited.json --output-result result.json \
  --report production.html
```

Repeat offline without any credential:

```bash
qqa tex --spec examples/models/regional_production.json \
  --solver auto --device auto --show-model \
  --output-result regional-production-result.json \
  --report regional-production-report.html
```